# 01 — Data Preparation: FigShare Brain Tumor Dataset

Converts the FigShare Brain Tumor Dataset raw `.mat` files into the shared project layout:

```
Drive/Senior_Project/data/figshare/processed/
├── images/<image_id>.png       grayscale uint8 256×256
├── masks/<image_id>.png        binary    uint8 256×256, values {0, 255}
├── metadata.csv                one row per image
├── metadata_summary.csv        per-class summary
└── preprocessing_config.json   the exact knobs this run used
```

**Dataset:** 3,064 single-slice `.mat` samples, 233 patients, 3 classes: meningioma / glioma / pituitary.

NB01 writes **directly to Drive** — it's a one-time write per dataset, not in the heavy-write training hot path.

Next: run **`02_split.ipynb`** to generate 5-fold CV CSVs.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q kagglehub h5py opencv-python-headless pillow numpy pandas matplotlib tqdm

## Cell 2 — Bootstrap: mount Drive + clone/pull repo

In [ ]:
import os, sys

# Bootstrap shim: clone the repo if missing, then add to sys.path.
# After this, src.* imports work and notebook_setup takes over.
if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)

# NB01 exception: write directly to Drive (one-time, not in training hot path).
PROJECT_ROOT = DRIVE_ROOT

print(f"DRIVE_ROOT:   {DRIVE_ROOT}")
print(f"REPO_ROOT:    {REPO_ROOT}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

## Cell 3 — Configuration

Adjust these knobs before running. No other cells need editing.

In [ ]:
KAGGLEHUB_DATASET_ID = "ashkhagan/figshare-brain-tumor-dataset"
TARGET_SIZE          = (256, 256)
NORMALIZATION        = "percentile"  # or "minmax"
RANDOM_SEED          = 42

print(f"KAGGLEHUB_DATASET_ID = {KAGGLEHUB_DATASET_ID}")
print(f"TARGET_SIZE          = {TARGET_SIZE}")
print(f"NORMALIZATION        = {NORMALIZATION}")
print(f"RANDOM_SEED          = {RANDOM_SEED}")

## Cell 4 — Resolve dataset paths

In [ ]:
from src.file_utils import dataset_paths

paths = dataset_paths(PROJECT_ROOT, "figshare")
for k, v in paths.items():
    print(f"  {k}: {v}")

## Cell 5 — Download raw `.mat` files via KaggleHub

KaggleHub caches in `/root/.cache/kagglehub/` (Colab local SSD). The download is fast on local SSD; we then process from there directly without copying to Drive's `raw/` (it would be a slow Drive write of ~870 MB and the kagglehub cache is reproducible from the dataset id alone).

In [ ]:
import kagglehub

download_path = kagglehub.dataset_download(KAGGLEHUB_DATASET_ID)
print(f"Downloaded to: {download_path}")

## Cell 6 — Discover `.mat` samples

In [ ]:
from src.preprocess_utils import discover_mat_files

mat_files = discover_mat_files(download_path)
n_candidates = len(mat_files)
print(f"Found {n_candidates} .mat samples")
assert n_candidates == 3064, f"expected 3,064 figshare .mat samples, found {n_candidates}"
print(f"First 3: {[p.name for p in mat_files[:3]]}")
print(f"Last 3:  {[p.name for p in mat_files[-3:]]}")
print("count sanity: OK")

## Cell 7 — Convert each `.mat` to PNG + collect metadata records

This is the slow cell — ~3,000 conversions, each writes 2 PNGs to Drive. Expect a few minutes.

In [ ]:
from tqdm.auto import tqdm
from src.preprocess_utils import convert_figshare_mat_to_png_record

records = []
for mat_path in tqdm(mat_files, desc="Converting"):
    rec = convert_figshare_mat_to_png_record(
        mat_path=mat_path,
        image_out_dir=paths["images"],
        mask_out_dir=paths["masks"],
        target_size=TARGET_SIZE,
        normalization=NORMALIZATION,
        path_style="relative",
        project_root=PROJECT_ROOT,
    )
    records.append(rec)

print(f"Converted {len(records)} samples")

## Cell 8 — Build and save `metadata.csv`

In [ ]:
import pandas as pd

metadata_df = pd.DataFrame(records)
print(f"metadata_df shape: {metadata_df.shape}")
print()
print(metadata_df.head())

metadata_df.to_csv(paths["metadata_csv"], index=False)
print(f"\nSaved {paths['metadata_csv']}")

## Cell 9 — Save `preprocessing_config.json`

In [ ]:
from datetime import datetime, timezone
from src.file_utils import save_json

config = {
    "dataset":              "figshare",
    "kagglehub_dataset_id": KAGGLEHUB_DATASET_ID,
    "target_size":          list(TARGET_SIZE),
    "normalization":        NORMALIZATION,
    "mask_threshold":       127,
    "image_format":         "grayscale_uint8_png",
    "mask_format":          "binary_uint8_png_0_255",
    "n_samples":            int(len(metadata_df)),
    "random_seed":          RANDOM_SEED,
    "generated_at":         datetime.now(timezone.utc).isoformat(),
}
save_json(config, paths["preprocessing_config_json"])
print(f"Saved {paths['preprocessing_config_json']}")

## Cell 10 — Per-class summary table

In [ ]:
summary = (
    metadata_df.groupby("tumor_class")
    .agg(
        n_images=("image_id", "count"),
        n_patients=("patient_id", "nunique"),
        mean_mask_area_ratio=("mask_area_ratio", "mean"),
        median_mask_area_ratio=("mask_area_ratio", "median"),
    )
    .reset_index()
)
print(summary)

summary.to_csv(paths["metadata_summary_csv"], index=False)
print(f"\nSaved {paths['metadata_summary_csv']}")

## Cell 11 — Visual QA: random examples per class

In [ ]:
from src.vis_utils import show_class_examples

qa_save_path = paths["figures_dataprep"] / "class_examples_grid.png"
show_class_examples(
    metadata_df=metadata_df,
    project_root=PROJECT_ROOT,
    n_per_class=3,
    random_state=RANDOM_SEED,
    save_path=qa_save_path,
)
print(f"\nSaved QA figure: {qa_save_path}")

## Done.

You now have, on Drive:

```
data/figshare/processed/
├── images/<image_id>.png
├── masks/<image_id>.png
├── metadata.csv                 (3,064 rows — meningioma / glioma / pituitary)
├── metadata_summary.csv
└── preprocessing_config.json
outputs/figures/data_preparation/figshare/class_examples_grid.png
```

Next: **NB02 (`02_split.ipynb`)** — generate 5-fold CSVs (`patient_level` and `image_level` schemes).